# Step 10 — Stratified pair-based sampling of ground_truth candidates

> Implements the paper's stratified pair-based purposive sampling: for each of the 19 selected smells, build a positive pool (`instance_count = 1` for the target smell) and a negative pool (target smell not flagged), each filtered to workflows with `instance_count ≤ 1` for *all* smells. Rank by ascending `total_unique_smells` and descending `workflow_run_count`. Apply the LOC cap from Step 09b (Tukey IQR upper fence) and stratify across LOC quartiles. Sample 100 positive and 100 negative candidates per smell (paper target = 50 confirmed-correct labels per polarity; oversample factor = 2 to absorb FP/FN during manual annotation).

**Paper section:** Step 3 — Ground-truth construction / Stratified sampling.

## Inputs
| Path | Format | Source |
|---|---|---|
| `data/9_ground_truth_candidates.csv` | CSV with smell features | Step 09 |
| `data/9b_loc_cap.json` | JSON (LOC cap, Tukey IQR upper fence) | Step 09b |
| `data/downloaded_workflows/` | Directory of YAML files (used for LOC counting + content embedding) | Step 8 (mined corpus) |

## Outputs
| Path | Format | Consumer |
|---|---|---|
| `data/10_ground_truth_sample.json` | JSON (sampled (workflow, smell) pairs with embedded YAML content) | Steps 10b, 11, 12, 12b (annotation + analysis) |
| `data/step10_report.json` | JSON | Step 11 |



In [1]:
# --- Notebook bootstrap ---
# Idempotent: chdir to repo root + reset sys.argv for argparse.
# Works in Jupyter/JupyterLab/Antigravity/VS Code/plain python on Linux+macOS+Windows.
from dataset_utils.notebook_bootstrap import setup

setup()

print("Step 10 — starting: stratified pair-based sampling...", flush=True)
"""Step 10: Pair-based ground_truth sampling from ground_truth candidates.

Design (decided 2026-04-25; oversample addendum 2026-05-08):
  Each behavioral smell receives N positives + N negatives candidates,
  where N = N_PER_POLARITY * OVERSAMPLE_FACTOR (default 50 * 2 = 100).
  N_PER_POLARITY is the paper-stated target of confirmed-correct labels;
  OVERSAMPLE_FACTOR=2 inflates the annotation pool to absorb FP/FN.
  Positives: file has the smell with instance_count == 1 AND every behavioral
             smell in the file has instance_count <= 1 (strict).
  Negatives: file does NOT have the smell (pure absence) AND passes strict.

The ground_truth is file-based (one entry per unique file). Each entry carries:
  - expected_labels[]      Behavioral smells present in the file (truth).
  - presentational_smells  Recorded but not evaluated.
  - evaluation_targets[]   Smells the agent must be queried about for this file.
                           Polarity is derived: smell ∈ expected_labels → positive,
                           else → negative.

Unit of evaluation: (file, smell) pair. For each pair, the agent's prediction
is compared against expected_labels-derived truth, populating TP/FP/TN/FN.

Sampling per smell pool (positives and negatives independently):
  0. Apply LOC cap from data/9b_loc_cap.json (Step 09b Tukey IQR upper fence) —
     drop rows with file_line_count > cap. Bounds outlier influence and
     LLM token cost during downstream agent evaluation.
  1. Sort by (total_unique_smells ASC, workflow_run_count DESC, file_key ASC).
     Fewer co-occurring smells reduces annotation noise; active workflows
     are prioritized; file_key breaks ties deterministically.
  2. Take pool of (LOC_QUARTILE_POOL_MULTIPLIER * N) top-ranked candidates.
  3. Partition by LOC quartile (Q1 / Q2 / Q3 / Q4) over the capped pool.
  4. Draw N/4 from each quartile, redistributing slots when a quartile is short.

Files used as a positive for one smell may serve as a negative for others —
the resulting evaluation_targets[] accumulates roles per file (file reuse OK).

Inputs:
  data/9_ground_truth_candidates.csv          Output of Step 09
  data/9b_loc_cap.json                  LOC cap (Step 09b Tukey IQR upper fence)
  data/downloaded_workflows/             YAML workflow corpus (LOC + content embedding)

Outputs:
  data/10_ground_truth_sample.json             File-based ground_truth (deduplicated)
  data/step10_report.json                Per-smell stats + sampling config
  data/tables/ground_truth_per_smell.tex       Auto-generated LaTeX table
  data/tables/ground_truth_summary.tex         Auto-generated LaTeX summary
"""

import argparse
import json
import statistics
from pathlib import Path

import polars as pl
from dataset_utils.ground_truth_sampling import (
    BEHAVIORAL_SMELL_ORDER,
    LOC_QUARTILE_POOL_MULTIPLIER,
    N_DELIVERED_PER_POLARITY,
    N_PER_POLARITY,
    OVERSAMPLE_FACTOR,
    PRESENTATIONAL_SMELLS,
    SMELL_CATALOG,
    _build_entry,
    _build_sort_key,
    _passes_strict,
    _select_loc_balanced,
    read_yaml,
)

# -- Main sampling logic -------------------------------------------------------

def sample_ground_truth(
    candidates_path: Path,
    output_path: Path,
    report_path: Path,
    n: int = N_PER_POLARITY,
    tables_dir: Path | None = None,
) -> dict:
    import sys as _sys
    try:
        _sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
    except NameError:
        pass  # Jupyter: __file__ undefined; dataset_utils on path via wheel install
    from dataset_utils.integrity import assert_input_matches_report
    from dataset_utils.latex import write_ground_truth_per_smell, write_ground_truth_summary

    input_sha = assert_input_matches_report(
        input_path=candidates_path,
        prior_report_path=Path("data/step9_report.json"),
        label="step 10 input",
    )

    # Load LOC cap from Step 09b. Verify chain integrity against step9b_report.json.
    from dataset_utils.hashing import compute_file_hash

    loc_cap_path = Path("data/9b_loc_cap.json")
    step9b_report_path = Path("data/step9b_report.json")
    if not loc_cap_path.exists() or not step9b_report_path.exists():
        msg = (
            f"step 10: LOC cap inputs missing. "
            f"Expected {loc_cap_path} and {step9b_report_path}. Run Step 09b first."
        )
        raise FileNotFoundError(msg)

    loc_cap_sha = compute_file_hash(loc_cap_path)
    with step9b_report_path.open("r", encoding="utf-8") as _rf:
        _step9b_report = json.load(_rf)
    expected_cap_sha = _step9b_report.get("output_sha256")
    if expected_cap_sha and expected_cap_sha != loc_cap_sha:
        msg = (
            f"step 10: data/9b_loc_cap.json SHA drift. "
            f"current={loc_cap_sha}, expected={expected_cap_sha} "
            f"(from {step9b_report_path}). Re-run Step 09b."
        )
        raise RuntimeError(msg)

    with loc_cap_path.open("r", encoding="utf-8") as _cf:
        loc_cap_payload = json.load(_cf)
    loc_cap = int(loc_cap_payload["cap"])
    print(f"  LOC cap from Step 09b: {loc_cap} (sha {loc_cap_sha[:16]}...)", flush=True)

    print("  reading input CSV (may take a moment for large files)...", flush=True)
    df = pl.read_csv(candidates_path, encoding="utf-8-sig")
    for col in ("total_unique_smells", "file_line_count", "workflow_run_count"):
        if df[col].dtype == pl.Utf8:
            df = df.with_columns(pl.col(col).cast(pl.Int64, strict=False).fill_null(0))

    # Apply LOC cap earliest: drop rows above the cap before strict-pool building,
    # so per-smell pools, quartile cutpoints, and report stats all reflect the
    # capped universe. Matches paper claim: "LOC ranges over sampled workflows".
    rows_before_cap = df.height
    df = df.filter(pl.col("file_line_count") <= loc_cap)
    rows_after_cap = df.height
    loc_cap_excluded = rows_before_cap - rows_after_cap
    print(
        f"  LOC cap applied: {loc_cap_excluded:,} rows dropped "
        f"({loc_cap_excluded/max(rows_before_cap,1)*100:.2f}%); {rows_after_cap:,} retained.",
        flush=True,
    )

    # Build per-row record + parsed smells.
    rows: list[dict] = []
    rows_by_key: dict[str, dict] = {}
    smells_by_key: dict[str, list[dict]] = {}
    for r in df.iter_rows(named=True):
        smells = json.loads(r["smells_json"])
        rows.append(r)
        rows_by_key[r["file_key"]] = r
        smells_by_key[r["file_key"]] = smells

    # Filter to strict pool.
    strict_keys: list[str] = []
    rejected_strict = 0
    for r in rows:
        if _passes_strict(smells_by_key[r["file_key"]]):
            strict_keys.append(r["file_key"])
        else:
            rejected_strict += 1

    # Index strict pool: smell -> set of file_keys with that smell present.
    strict_smell_presence: dict[str, set[str]] = {s: set() for s in BEHAVIORAL_SMELL_ORDER}
    for fk in strict_keys:
        for s in smells_by_key[fk]:
            pascal = s.get("pascal")
            if pascal in strict_smell_presence:
                strict_smell_presence[pascal].add(fk)

    # Per-smell sampling.
    per_smell_stats: dict[str, dict] = {}
    selections: dict[str, dict[str, list[str]]] = {}  # smell -> {"positive": [keys], "negative": [keys]}

    for pascal in BEHAVIORAL_SMELL_ORDER:
        present_keys = strict_smell_presence[pascal]
        absent_keys = set(strict_keys) - present_keys

        pos_candidates = [rows_by_key[k] for k in present_keys]
        neg_candidates = [rows_by_key[k] for k in absent_keys]

        pos_candidates.sort(key=_build_sort_key)
        neg_candidates.sort(key=_build_sort_key)

        n_delivered = n * OVERSAMPLE_FACTOR
        pos_selected = _select_loc_balanced(pos_candidates, n_delivered)
        neg_selected = _select_loc_balanced(neg_candidates, n_delivered)

        pos_locs = [r["file_line_count"] or 0 for r in pos_selected]
        neg_locs = [r["file_line_count"] or 0 for r in neg_selected]

        paper_id, family, _ = SMELL_CATALOG[pascal]
        per_smell_stats[pascal] = {
            "id": paper_id,
            "family": family,
            "available_pos_strict": len(pos_candidates),
            "available_neg_strict": len(neg_candidates),
            "delivered_pos": len(pos_selected),
            "delivered_neg": len(neg_selected),
            "selected_pos": len(pos_selected),
            "selected_neg": len(neg_selected),
            "loc_min_pos": min(pos_locs) if pos_locs else None,
            "loc_max_pos": max(pos_locs) if pos_locs else None,
            "loc_median_pos": int(statistics.median(pos_locs)) if pos_locs else None,
            "loc_min_neg": min(neg_locs) if neg_locs else None,
            "loc_max_neg": max(neg_locs) if neg_locs else None,
            "loc_median_neg": int(statistics.median(neg_locs)) if neg_locs else None,
        }
        # Tuples (priority, file_key) — priority 1..N_DELIVERED in primary-rank order.
        selections[pascal] = {
            "positive": [(i + 1, r["file_key"]) for i, r in enumerate(pos_selected)],
            "negative": [(i + 1, r["file_key"]) for i, r in enumerate(neg_selected)],
        }

    # Build deduplicated ground_truth entries with evaluation_targets accumulated.
    ground_truth_entries: dict[str, dict] = {}
    yaml_cache: dict[str, str | None] = {}
    skipped_no_yaml: list[str] = []

    for pascal, splits in selections.items():
        for polarity_key in ("positive", "negative"):
            for priority, fk in splits[polarity_key]:
                if fk not in ground_truth_entries:
                    if fk not in yaml_cache:
                        yaml_cache[fk] = read_yaml(fk)
                    yaml_content = yaml_cache[fk]
                    if yaml_content is None:
                        skipped_no_yaml.append(fk)
                        continue
                    ground_truth_entries[fk] = _build_entry(
                        rows_by_key[fk], yaml_content, smells_by_key[fk]
                    )
                existing = {t["pascal"] for t in ground_truth_entries[fk]["evaluation_targets"]}
                if pascal not in existing:
                    ground_truth_entries[fk]["evaluation_targets"].append({
                        "pascal": pascal,
                        "polarity": polarity_key,
                        "priority": priority,
                    })

    # Sort each entry's evaluation_targets: paper_id ASC, then polarity (positive
    # before negative) for deterministic JSON output. Priority is intra-bucket.
    for entry in ground_truth_entries.values():
        entry["evaluation_targets"].sort(
            key=lambda t: (SMELL_CATALOG[t["pascal"]][0], t["polarity"])
        )

    # Final ground_truth: list sorted by file_key.
    ground_truth_list = [ground_truth_entries[k] for k in sorted(ground_truth_entries.keys())]

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(ground_truth_list, f, indent=2, ensure_ascii=False)

    # Per-smell coverage table for LaTeX (drops loc_* columns per Q26).
    coverage_table = []
    for pascal in BEHAVIORAL_SMELL_ORDER:
        stats = per_smell_stats[pascal]
        _, family, latex_name = SMELL_CATALOG[pascal]
        coverage_table.append({
            "id": stats["id"],
            "family": family,
            "smell": pascal,
            "latex_name": latex_name,
            "detected": stats["available_pos_strict"],
            "selected": stats["selected_pos"] + stats["selected_neg"],
        })

    per_smell_tex_path: Path | None = None
    summary_tex_path: Path | None = None
    if tables_dir is not None:
        per_smell_tex_path = tables_dir / "ground_truth_per_smell.tex"
        write_ground_truth_per_smell(coverage_table, per_smell_tex_path)

    n_delivered = n * OVERSAMPLE_FACTOR
    full_pos = sum(1 for s in per_smell_stats.values() if s["selected_pos"] >= n_delivered)
    full_neg = sum(1 for s in per_smell_stats.values() if s["selected_neg"] >= n_delivered)
    target_pos = sum(1 for s in per_smell_stats.values() if s["selected_pos"] >= n)
    target_neg = sum(1 for s in per_smell_stats.values() if s["selected_neg"] >= n)

    report = {
        "n_per_polarity": n,
        "n_target_per_polarity": n,
        "n_delivered_per_polarity": n_delivered,
        "oversample_factor": OVERSAMPLE_FACTOR,
        "smells_meeting_target_pos": target_pos,
        "smells_meeting_target_neg": target_neg,
        "total_ground_truth_entries": len(ground_truth_list),
        "unique_workflow_files": len(ground_truth_list),
        "unique_smells_targeted": len(BEHAVIORAL_SMELL_ORDER),
        "smells_with_full_pos": full_pos,
        "smells_with_full_neg": full_neg,
        "smells_with_full_coverage": min(full_pos, full_neg),
        "smells_with_partial_coverage": (
            len(BEHAVIORAL_SMELL_ORDER) - min(full_pos, full_neg)
        ),
        "smells_with_no_coverage": sum(
            1 for s in per_smell_stats.values()
            if s["selected_pos"] == 0 or s["selected_neg"] == 0
        ),
        "presentational_smells": sorted(PRESENTATIONAL_SMELLS),
        "skipped_no_yaml": skipped_no_yaml,
        "sampling_config": {
            "algorithm": "pure-absence pair-based with strict file-wide filter, LOC-capped, quartile-stratified, 2x-oversampled",
            "n_target_per_polarity": n,
            "n_delivered_per_polarity": n_delivered,
            "oversample_factor": OVERSAMPLE_FACTOR,
            "loc_quartile_pool_multiplier": LOC_QUARTILE_POOL_MULTIPLIER,
            "loc_cap_used": loc_cap,
            "loc_cap_source": str(loc_cap_path),
            "loc_cap_sha256": loc_cap_sha,
            "loc_cap_excluded": loc_cap_excluded,
            "rows_before_loc_cap": rows_before_cap,
            "rows_after_loc_cap": rows_after_cap,
            "strict_filter": "every behavioral smell instance_count <= 1",
            "rejected_strict": rejected_strict,
            "strict_pool_size": len(strict_keys),
        },
        "per_smell_coverage_table": coverage_table,
        "per_smell": per_smell_stats,
    }

    if tables_dir is not None:
        summary_tex_path = tables_dir / "ground_truth_summary.tex"
        write_ground_truth_summary(report, summary_tex_path)

    from dataset_utils.hashing import compute_file_hash
    report["input_sha256"] = input_sha
    if output_path.exists():
        report["output_file"] = str(output_path)
        report["output_sha256"] = compute_file_hash(output_path)
    if per_smell_tex_path is not None and per_smell_tex_path.exists():
        report["ground_truth_per_smell_tex"] = str(per_smell_tex_path)
        report["ground_truth_per_smell_tex_sha256"] = compute_file_hash(per_smell_tex_path)
    if summary_tex_path is not None and summary_tex_path.exists():
        report["ground_truth_summary_tex"] = str(summary_tex_path)
        report["ground_truth_summary_tex_sha256"] = compute_file_hash(summary_tex_path)

    with report_path.open("w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, ensure_ascii=False)

    return report

# -- Console output ------------------------------------------------------------

def _print_coverage_table(report: dict, n: int) -> None:
    table = report["per_smell_coverage_table"]
    per_smell = report["per_smell"]

    print()
    print(
        f"  Per-smell sampling coverage "
        f"(target N={n} confirmed-correct + delivered {n * OVERSAMPLE_FACTOR} per polarity, oversample {OVERSAMPLE_FACTOR}x):"
    )
    print(f"  {'ID':>3}  {'Smell':<55} {'Pos pool':>9} {'Sel+':>5} {'Neg pool':>9} {'Sel-':>5}")
    print(f"  {'---':>3}  {'-' * 55} {'-' * 9} {'-' * 5} {'-' * 9} {'-' * 5}")

    current_family = None
    for row in table:
        if row["family"] != current_family:
            current_family = row["family"]
            print(f"  {'':>3}  {current_family}")
        s = per_smell[row["smell"]]
        print(
            f"  {row['id']:>3}  {row['smell'][:55]:<55}"
            f" {s['available_pos_strict']:>9,} {s['selected_pos']:>5}"
            f" {s['available_neg_strict']:>9,} {s['selected_neg']:>5}"
        )

    cfg = report["sampling_config"]
    print()
    print(f"  Strict pool      : {cfg['strict_pool_size']:,}")
    print(f"  Rejected strict  : {cfg['rejected_strict']:,}")
    print(f"  Total entries    : {report['total_ground_truth_entries']:,}")
    print(f"  Skipped (no yaml): {len(report['skipped_no_yaml'])}")
    n_d = report.get("n_delivered_per_polarity", n)
    print(f"  Full pos (≥{n_d})  : {report['smells_with_full_pos']}/{report['unique_smells_targeted']}")
    print(f"  Full neg (≥{n_d})  : {report['smells_with_full_neg']}/{report['unique_smells_targeted']}")
    print(f"  Meet target pos (≥{n}): {report.get('smells_meeting_target_pos', 0)}/{report['unique_smells_targeted']}")
    print(f"  Meet target neg (≥{n}): {report.get('smells_meeting_target_neg', 0)}/{report['unique_smells_targeted']}")

# -- CLI -----------------------------------------------------------------------

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Pair-based ground_truth sampling.")
    parser.add_argument(
        "--input", type=Path, default=Path("data/9_ground_truth_candidates.csv"),
        help="Input ground_truth candidates CSV (output of step 09).",
    )
    parser.add_argument(
        "--output", type=Path, default=Path("data/10_ground_truth_sample.json"),
        help="Output ground_truth sample JSON.",
    )
    parser.add_argument(
        "--report", type=Path, default=Path("data/step10_report.json"),
        help="Output report JSON.",
    )
    parser.add_argument(
        "--tables-dir", type=Path, default=Path("data/tables"),
        help="Output directory for auto-generated LaTeX tables. "
             "Set to empty string to skip.",
    )
    parser.add_argument(
        "--n", type=int, default=N_PER_POLARITY,
        help=f"Target N (paper-stated confirmed-correct labels) per polarity per smell. "
             f"Delivered = N * OVERSAMPLE_FACTOR ({OVERSAMPLE_FACTOR}x = {N_DELIVERED_PER_POLARITY}). "
             f"Default: {N_PER_POLARITY}.",
    )
    parser.add_argument(
        "--latex", type=Path, default=None,
        help="Optional: copy ground_truth_per_smell.tex to this path after generation "
             "(typically the paper's tables/ directory).",
    )
    return parser.parse_args()

def main() -> None:
    args = parse_args()
    tables_dir = args.tables_dir if str(args.tables_dir) else None
    report = sample_ground_truth(
        args.input, args.output, args.report, args.n, tables_dir=tables_dir,
    )

    if args.latex is not None and tables_dir is not None:
        src = tables_dir / "ground_truth_per_smell.tex"
        if src.exists():
            args.latex.parent.mkdir(parents=True, exist_ok=True)
            args.latex.write_text(src.read_text(encoding="utf-8"), encoding="utf-8")

    print("Ground truth sample generated.")
    print(f"  Input        : {args.input}")
    print(f"  Output       : {args.output}")
    print(f"  Report       : {args.report}")
    if tables_dir:
        print(f"  Tables dir   : {tables_dir}")
    if args.latex:
        print(f"  Paper LaTeX  : {args.latex}")
    print(f"  N/polarity   : {args.n}")

    _print_coverage_table(report, args.n)

main()


Step 10 — starting: stratified pair-based sampling...


  LOC cap from Step 09b: 209 (sha 3dfeb6a1e48dd19e...)


  reading input CSV (may take a moment for large files)...


  LOC cap applied: 4,716 rows dropped (9.46%); 45,113 retained.


Ground truth sample generated.
  Input        : data/9_ground_truth_candidates.csv
  Output       : data/10_ground_truth_sample.json
  Report       : data/step10_report.json
  Tables dir   : data/tables
  N/polarity   : 50

  Per-smell sampling coverage (target N=50 confirmed-correct + delivered 100 per polarity, oversample 2x):
   ID  Smell                                                    Pos pool  Sel+  Neg pool  Sel-
  ---  ------------------------------------------------------- --------- ----- --------- -----
       Security
    1  MissingPermExternalActions                                  6,336   100     8,042   100
    2  UnpinnedActionVersion                                       4,987   100     9,391   100
    3  MissingTokenPermissions                                       663   100    13,715   100
       Performance/Optimization
    4  MissingForkGuardPR                                          1,998   100    12,380   100
    5  MissingJobTimeout                           

---

**Next step:** [`11_summarize_evaluation.ipynb`](./11_summarize_evaluation.ipynb)
